# Training regimes
Saved evidence only. No training or network access.

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
study = json.loads(Path(study_path).read_text())
print(study['status'], study['limits'])
anchor = 'scripted-greedy-fixed-anchor' if study['study'] in {'value-token-screen', 'pooling-filter', 'history-input', 'depth-screen'} else 'random-smoke-anchor'
rows = [r for r in study['measurements'] if r['opponent'] == anchor and r.get('phase', 'development') == 'development']
fig, ax = plt.subplots(figsize=(8,4))
for name, seed, variant in sorted({(r['regime'], r['seed'], r.get('variant', 'raw')) for r in rows}):
    points = sorted([r for r in rows if (r['regime'], r['seed'], r.get('variant', 'raw')) == (name, seed, variant) and r['complete'] and r['score'] is not None], key=lambda r: r['training_seconds'])
    ax.step([r['training_seconds']/3600 for r in points], [r['score'] for r in points], where='post', marker='o', label=f'{name} seed {seed} {variant}')
ax.set(xlabel='Actual cumulative training hours', ylabel=f'Score vs {anchor}', ylim=(0,1))
if rows: ax.legend()
fig.tight_layout()
fig.savefig('learning-curves.png')
plt.show()
if study['study'] in {'value-token-screen', 'pooling-filter', 'history-input', 'depth-screen'}:
    fig, ax = plt.subplots(figsize=(8,4))
    for name, seed in sorted({(r['regime'], r['seed']) for r in rows}):
        points = sorted([r for r in rows if r['regime'] == name and r['seed'] == seed and r['complete']], key=lambda r: r['learner_transitions'])
        ax.plot([r['learner_transitions'] for r in points], [r['score'] for r in points], marker='o', label=f'{name} seed {seed}')
    ax.set(xlabel='Cumulative learner transitions', ylabel=f'Score vs {anchor}', ylim=(0,1))
    if rows: ax.legend()
    fig.tight_layout()
    fig.savefig('learning-transitions.png')
    plt.show()
print('Training seeds:', study.get('seeds', [study.get('seed')]), '; small cohorts are descriptive only.')
for item in study['runs']:
    if not Path(item['path']).exists():
        print(item, 'Run export unavailable: failure before persistence')
        continue
    run = json.loads(Path(item['path']).read_text())
    for stage in run['stages']:
        print(run['regime']['id'], stage['id'], stage['status'], 'updates', len(stage['diagnostics']))


In [ ]:
comparison = json.loads((Path(study_path).parent / 'cost-comparison.json').read_text())
print('Common observed cost window:', comparison)
if comparison['status'] == 'available' and study['study'] == 'ataraxos-ablations':
    points = comparison['rows']
    controls = {(p['seed'], p['variant']): p['score'] for p in points if p['regime'] == 'rl-control'}
    fig, ax = plt.subplots(figsize=(9,4))
    ax.scatter([p['regime'] for p in points], [p['score'] - controls[(p['seed'], p['variant'])] for p in points])
    ax.axhline(0, color='gray')
    ax.set(ylabel='Score difference vs control at shared cost', title='Paired seed effects: descriptive evidence only')
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig('ablation-effects.png')
    plt.show()
print('S1-S5: unavailable; current-world scenario premises not yet certified.')
print('Raw/EMA: distinct artifact identities, correlated outputs within a training seed; never additional replicates.')


In [ ]:
for name in ('anchor-cost-comparisons.json', 'cost-cutoffs.json', 'uncertainty.json'):
    print(name, json.loads((Path(study_path).parent / name).read_text()))
print('Development and endpoint cohorts are separate; checkpoint-index matches do not imply equal cost.')


In [ ]:
capacity_path = Path(study_path).parent / 'capacity-analysis.json'
if study['study'] == 'model-capacity':
    capacity = json.loads(capacity_path.read_text())
    print(capacity['decision'], capacity['limits'])
    print('Early area:', capacity['early_area'])
    print('Scheduled crossings:', capacity['crossings'])
    print('Held-out terminal observations:', capacity['terminal'])
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (axis, curve) in zip(axes, capacity['curves'].items()):
        ax.set(xlabel=axis, ylabel='Score vs random anchor', ylim=(0, 1))
        if curve['status'] != 'available':
            ax.set_title('Unavailable')
            continue
        start, end = curve['start'], curve['end']
        for item in curve['rows']:
            points = sorted([p for p in curve['points'] if p['regime'] == item['regime'] and p['seed'] == item['seed'] and p['complete'] and p['score'] is not None], key=lambda p: p['training_seconds'])
            left = max((p for p in points if p['training_seconds'] <= start), key=lambda p: p['training_seconds'])
            inside = [p for p in points if start < p['training_seconds'] <= end]
            ax.step([start] + [p['training_seconds'] for p in inside] + [end], [left['score']] + [p['score'] for p in inside] + [item['score']], where='post', label=f"{item['regime']} / {item['seed']}")
        ax.legend(fontsize=6)
    fig.tight_layout()
    fig.savefig('capacity-curves.png')
    plt.show()


In [ ]:
if study['study'] in {'history-input', 'depth-screen'}:
    contrast_path = Path(study_path).parent / 'history-contrast.json'
    if contrast_path.exists():
        print(json.loads(contrast_path.read_text()))
    print('Two scheduled checkpoints cannot resolve detailed learning dynamics. Exposure is post-treatment accounting; the frozen protocol identifies the treatment.')
    if study['status'] == 'completed':
        for axis, label in [('updates', 'Collect/update iterations'), ('learner_transitions', 'Learner transitions'), ('decisions', 'Native decisions'), ('optimizer_exposures', 'Optimizer exposures')]:
            fig, ax = plt.subplots(figsize=(8, 4))
            for name, seed in sorted({(r['regime'], r['seed']) for r in rows}):
                points = sorted([{**r, 'updates': r['learner_transitions'] // 256} for r in rows if r['regime'] == name and r['seed'] == seed], key=lambda r: r[axis])
                ax.step([r[axis] for r in points], [r['score'] for r in points], where='post', marker='o', label=f'{name} seed {seed}')
            ax.set(xlabel=label, ylabel=f'Score vs {anchor}')
            ax.legend()
            fig.tight_layout()
            fig.savefig(f'history-{axis}.png')
            plt.show()
    for metric in ['policy_loss', 'value_loss', 'advantage_mean', 'advantage_abs_mean', 'retained', 'optimizer_exposures', 'actor_exposures', 'critic_exposures', 'learning_rate', 'tau']:
        fig, ax = plt.subplots(figsize=(8, 4))
        for item in study['runs']:
            path = Path(item['path'])
            if not path.exists():
                continue
            run = json.loads(path.read_text())
            diagnostics = [d for stage in run['stages'] for d in stage['diagnostics']]
            points = [(i + 1, d[metric]) for i, d in enumerate(diagnostics) if isinstance(d.get(metric), (int, float))]
            if points:
                ax.plot([p[0] for p in points], [p[1] for p in points], label=f"{run['regime']['id']} seed {run['seed']}")
        ax.set(xlabel='Collect/update iteration (including empty-filter skips)', ylabel=metric)
        if ax.lines:
            ax.legend()
        fig.tight_layout()
        fig.savefig(f'history-{metric}.png')
        plt.show()
